# Regularization via Priors (Section 7.5)

**The idea**: instead of choosing *which* model to use (model selection), we can make any model less likely to overfit by using **skeptical priors** — priors that resist large coefficients.

## Setup

Same 6 polynomial models on the primate brain data. But now we fit each model under **3 different prior widths** on the slope coefficients β:

| Prior | β ~ Normal(0, σ) | Character |
|-------|-----------------|----------|
| Wide  | Normal(0, **10**) | Essentially flat — no regularization |
| Moderate | Normal(0, **1**) | Weakly informative |
| Skeptical | Normal(0, **0.2**) | Strongly regularizing |

**Prediction**:
- For simple models (degree 1-2): prior width barely matters — data dominates
- For complex models (degree 4-5): skeptical priors dramatically reduce overfitting

**Watch** how `p_WAIC` (effective parameters) responds — it's the fingerprint of regularization.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import logsumexp
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent))
from src.quap import quap

plt.style.use('default')
%matplotlib inline
np.random.seed(42)

print('✓ Imports loaded')

In [ ]:
# Primate brain data
data = pd.DataFrame({
    'species': ['afarensis', 'africanus', 'habilis', 'boisei',
                'rudolfensis', 'ergaster', 'sapiens'],
    'brain': [438, 452, 612, 521, 752, 871, 1350],
    'mass':  [37.0, 35.5, 34.5, 41.5, 55.5, 61.0, 53.5]
})

mass_std     = (data['mass'] - data['mass'].mean()) / data['mass'].std()
brain_scaled = data['brain'] / data['brain'].max()
x, y = mass_std.values, brain_scaled.values

print(f"N = {len(x)} species")

## Fitting with Configurable Priors

The only change from the previous notebook: `prior_sigma` controls how wide the β prior is.
Everything else stays identical — same likelihood, same data, same optimizer.

In [ ]:
def fit_polynomial(x, y, degree, prior_sigma=10.0):
    """
    Fit polynomial of given degree with configurable slope prior.

    prior_sigma controls β ~ Normal(0, prior_sigma):
      - Large  (10): essentially flat, no regularization
      - Medium  (1): weakly informative
      - Small (0.2): skeptical / strongly regularizing
    """
    X_poly = np.column_stack([x**i for i in range(1, degree + 1)])

    def neg_log_posterior(params):
        alpha     = params[0]
        betas     = params[1:degree + 1]
        log_sigma = params[degree + 1]
        sigma     = np.exp(log_sigma)

        mu = alpha + np.sum(X_poly * betas, axis=1)

        log_lik   = np.sum(stats.norm.logpdf(y, loc=mu, scale=sigma))
        log_prior = (
            stats.norm.logpdf(alpha, 0.5, 1) +
            np.sum(stats.norm.logpdf(betas, 0, prior_sigma)) +   # <-- the knob
            stats.expon.logpdf(sigma, scale=1)
        )
        return -(log_lik + log_prior + log_sigma)

    fit = quap(
        neg_log_posterior,
        [0.5] + [0] * degree + [np.log(0.5)],
        ['alpha'] + [f'beta{i}' for i in range(1, degree + 1)] + ['log_sigma']
    )
    fit.transform_param('log_sigma', 'sigma', np.exp)
    fit._prior_sigma = prior_sigma
    return fit


# Three prior widths to compare
PRIORS = {
    'wide (σ=10)':      10.0,
    'moderate (σ=1)':    1.0,
    'skeptical (σ=0.2)': 0.2,
}
DEGREES = list(range(1, 6))   # degrees 1-5; skip 6 (fixed-sigma special case)

print('✓ fit_polynomial() defined with configurable prior')
print(f'  Will fit {len(DEGREES)} degrees × {len(PRIORS)} priors = {len(DEGREES)*len(PRIORS)} models')

In [ ]:
# Fit all models (suppress per-model output for cleanliness)
import warnings
import io, contextlib

models = {}   # models[(degree, prior_label)] = fitted model

for prior_label, prior_sigma in PRIORS.items():
    for deg in DEGREES:
        with contextlib.redirect_stdout(io.StringIO()):   # suppress quap prints
            m = fit_polynomial(x, y, deg, prior_sigma=prior_sigma)
        models[(deg, prior_label)] = m

print(f'✓ Fitted {len(models)} models')
print('  Convergence summary:')
for (deg, pl), m in models.items():
    if not m.success:
        print(f'  ⚠️  Degree {deg}, {pl}: did not converge')

## Visualising the Fits

Focus on **degree 4 and 5** where the prior makes the biggest difference.
Each row = one degree; each column = one prior width.

In [ ]:
x_plot = np.linspace(x.min() - 0.3, x.max() + 0.3, 200)

def predict_mean(model, x_plot, degree, n_samples=2000):
    """Return posterior mean and 89% CI for the regression line."""
    samples   = model.sample(n=n_samples, seed=42)
    alpha_s   = samples['alpha'].values
    betas_s   = samples[[f'beta{i}' for i in range(1, degree + 1)]].values
    X_plot    = np.column_stack([x_plot**i for i in range(1, degree + 1)])
    mu_mat    = alpha_s[:, None] + betas_s @ X_plot.T   # (S, n_plot)
    return mu_mat.mean(axis=0), np.percentile(mu_mat, 5.5, axis=0), np.percentile(mu_mat, 94.5, axis=0)


focus_degrees = [3, 4, 5]
prior_labels  = list(PRIORS.keys())
colors        = ['tomato', 'steelblue', 'seagreen']

fig, axes = plt.subplots(len(focus_degrees), len(PRIORS),
                         figsize=(15, 4 * len(focus_degrees)), sharey=True)

for row, deg in enumerate(focus_degrees):
    for col, (prior_label, color) in enumerate(zip(prior_labels, colors)):
        ax  = axes[row, col]
        m   = models[(deg, prior_label)]
        mu, lo, hi = predict_mean(m, x_plot, deg)

        ax.fill_between(x_plot, lo, hi, alpha=0.2, color=color)
        ax.plot(x_plot, mu, '-', color=color, linewidth=2)
        ax.scatter(x, y, s=60, color='black', zorder=3, alpha=0.8)

        ax.set_ylim(-0.1, 1.3)
        ax.axhline(0, color='gray', lw=0.8, ls='--', alpha=0.4)
        ax.axhline(1, color='gray', lw=0.8, ls='--', alpha=0.4)
        ax.grid(True, alpha=0.25)

        if row == 0:
            ax.set_title(prior_label, fontsize=12, fontweight='bold', color=color)
        if col == 0:
            ax.set_ylabel(f'Degree {deg}\nbrain (scaled)', fontsize=11)
        if row == len(focus_degrees) - 1:
            ax.set_xlabel('body mass (std)', fontsize=10)

plt.suptitle('Effect of Prior Width on Polynomial Fits\n'
             '(shaded = 89% posterior interval for the mean)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Key pattern:')
print('  • Degree 3: all three priors give similar fits')
print('  • Degree 4-5: wide prior = wild wiggles; skeptical prior = smoother curves')

## WAIC for All 15 Models

Now let's quantify what we just saw visually.

In [ ]:
def compute_waic(model, x, y, degree, n_samples=10_000):
    """Compute WAIC using posterior samples from quap."""
    samples   = model.sample(n=n_samples, seed=42)
    X_poly    = np.column_stack([x**i for i in range(1, degree + 1)])
    alpha_s   = samples['alpha'].values
    betas_s   = samples[[f'beta{i}' for i in range(1, degree + 1)]].values
    sigma_s   = samples['sigma'].values
    mu_mat    = alpha_s[:, None] + betas_s @ X_poly.T
    log_lik   = stats.norm.logpdf(y[None, :], loc=mu_mat, scale=sigma_s[:, None])

    lppd_i   = logsumexp(log_lik, axis=0) - np.log(n_samples)
    p_waic_i = log_lik.var(axis=0)
    waic_i   = -2 * (lppd_i - p_waic_i)

    return dict(
        waic   = waic_i.sum(),
        lppd   = lppd_i.sum(),
        p_waic = p_waic_i.sum(),
        se     = np.sqrt(len(y) * waic_i.var()),
    )


# Compute WAIC for all models
waic_results = {}
for (deg, pl), m in models.items():
    with contextlib.redirect_stdout(io.StringIO()):
        waic_results[(deg, pl)] = compute_waic(m, x, y, deg)

# Build summary table
rows = []
for (deg, pl) in models:
    w = waic_results[(deg, pl)]
    rows.append(dict(Degree=deg, Prior=pl,
                     WAIC=round(w['waic'], 1),
                     lppd=round(w['lppd'], 2),
                     p_WAIC=round(w['p_waic'], 2),
                     SE=round(w['se'], 1)))

df = pd.DataFrame(rows).sort_values(['Degree', 'Prior'])
print('WAIC by degree and prior width:')
print('=' * 70)
print(df.to_string(index=False))
print('=' * 70)

## The Regularization Signal: p_WAIC

`p_WAIC` is the effective number of parameters — how many parameters the model *behaves as if* it has, from the data's perspective. Skeptical priors shrink this without necessarily shrinking the nominal parameter count.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
prior_labels = list(PRIORS.keys())
colors       = ['tomato', 'steelblue', 'seagreen']
markers      = ['o', 's', '^']

# ── Plot 1: WAIC vs degree, one line per prior ────────────────────────────────
ax = axes[0]
for pl, color, marker in zip(prior_labels, colors, markers):
    waic_vals = [waic_results[(d, pl)]['waic'] for d in DEGREES]
    ax.plot(DEGREES, waic_vals, marker=marker, color=color,
            linewidth=2, markersize=8, label=pl)
ax.set_xlabel('Polynomial Degree', fontsize=12)
ax.set_ylabel('WAIC (lower = better)', fontsize=12)
ax.set_title('WAIC by Degree & Prior\n', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# ── Plot 2: p_WAIC vs degree ──────────────────────────────────────────────────
ax = axes[1]
for pl, color, marker in zip(prior_labels, colors, markers):
    p_waic_vals = [waic_results[(d, pl)]['p_waic'] for d in DEGREES]
    ax.plot(DEGREES, p_waic_vals, marker=marker, color=color,
            linewidth=2, markersize=8, label=pl)

# Nominal parameter counts for reference
nominal = [d + 2 for d in DEGREES]   # alpha + betas + sigma
ax.plot(DEGREES, nominal, 'k--', linewidth=1.5, alpha=0.5, label='nominal params')

ax.set_xlabel('Polynomial Degree', fontsize=12)
ax.set_ylabel('p_WAIC (effective parameters)', fontsize=12)
ax.set_title('Effective Parameters by Prior\n(dashed = nominal count)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# ── Plot 3: lppd vs degree — does regularization hurt fit? ───────────────────
ax = axes[2]
for pl, color, marker in zip(prior_labels, colors, markers):
    lppd_vals = [waic_results[(d, pl)]['lppd'] for d in DEGREES]
    ax.plot(DEGREES, lppd_vals, marker=marker, color=color,
            linewidth=2, markersize=8, label=pl)
ax.set_xlabel('Polynomial Degree', fontsize=12)
ax.set_ylabel('lppd (in-sample fit, higher = better)', fontsize=12)
ax.set_title('In-Sample Fit by Prior\n(how much does regularization cost?)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('Regularization via Skeptical Priors: Effect on WAIC Components',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## WAIC Heatmap: The Full Picture

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
metrics = [('WAIC', 'WAIC\n(lower = better)', 'RdYlGn_r'),
           ('p_WAIC', 'p_WAIC\n(effective params)', 'YlOrRd'),
           ('lppd', 'lppd\n(fit, higher = better)', 'YlGn')]

for ax, (metric, label, cmap) in zip(axes, metrics):
    grid = np.array([
        [waic_results[(d, pl)][metric.lower()] for pl in prior_labels]
        for d in DEGREES
    ])

    im = ax.imshow(grid, cmap=cmap, aspect='auto')
    plt.colorbar(im, ax=ax, shrink=0.8)

    # Annotate cells
    for i in range(len(DEGREES)):
        for j in range(len(prior_labels)):
            ax.text(j, i, f'{grid[i,j]:.1f}',
                    ha='center', va='center', fontsize=10, fontweight='bold',
                    color='white' if abs(grid[i,j]) > grid.max()*0.6 else 'black')

    ax.set_xticks(range(len(prior_labels)))
    ax.set_xticklabels(['Wide\n(σ=10)', 'Moderate\n(σ=1)', 'Skeptical\n(σ=0.2)'], fontsize=9)
    ax.set_yticks(range(len(DEGREES)))
    ax.set_yticklabels([f'Degree {d}' for d in DEGREES])
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.set_xlabel('Prior width →  more regularization', fontsize=10)

plt.suptitle('Regularization Heatmaps: Degree × Prior Width',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Reduction in p_WAIC: Quantifying Regularization Strength

In [ ]:
# How much does the skeptical prior reduce p_WAIC compared to the wide prior?
wide_label     = 'wide (σ=10)'
skeptical_label = 'skeptical (σ=0.2)'

print('Reduction in effective parameters (p_WAIC) from wide → skeptical prior:')
print('=' * 60)
print(f'{"Degree":>8}  {"Wide":>8}  {"Skeptical":>10}  {"Reduction":>10}  {"% Drop":>8}')
print('-' * 60)
for deg in DEGREES:
    wide_p  = waic_results[(deg, wide_label)]['p_waic']
    skep_p  = waic_results[(deg, skeptical_label)]['p_waic']
    red     = wide_p - skep_p
    pct     = 100 * red / wide_p if wide_p > 0 else 0
    print(f'{deg:>8}  {wide_p:>8.2f}  {skep_p:>10.2f}  {red:>10.2f}  {pct:>7.1f}%')
print('=' * 60)
print('\n→ For low-degree models: small reduction (data already constrains well)')
print('→ For high-degree models: large reduction (prior does heavy lifting)')

## Key Insights

### 1. Prior width controls effective parameters, not nominal parameters

A degree-5 model always has 7 nominal parameters (α + 5β + σ). But its **effective** parameter count (`p_WAIC`) depends on the prior:
- Wide prior → p_WAIC ≈ nominal count (data can use all the flexibility)
- Skeptical prior → p_WAIC shrinks significantly (prior prevents extreme coefficients)

### 2. Low-degree models are robust to prior choice

For degree 1-2, all three priors give nearly identical fits. With 7 data points, even a wide prior `Normal(0, 10)` is constrained by the data for simple models. The prior only matters when the model has *more flexibility than the data can pin down*.

### 3. The bias-variance tradeoff, visually

| Prior | Variance (wiggliness) | Bias (systematic error) | lppd | p_WAIC |
|-------|-----------------------|--------------------------|------|--------|
| Wide  | High | Low | High | High |
| Skeptical | Low | Higher | Slightly lower | Low |

Skeptical priors **reduce variance at the cost of slight bias** — that's the regularization tradeoff. When the model is overparameterized, the bias cost is small and the variance benefit is large.

### 4. Regularization vs model selection

Rather than asking *"which model?"*, regularization asks *"how much should we trust the data vs our prior beliefs?"* A skeptical prior on a degree-5 model can outperform a flat prior on a degree-2 model — the prior shape matters as much as the model architecture.

### 5. Connection to ridge regression

`Normal(0, σ)` priors on coefficients are the Bayesian equivalent of **L2 / ridge regularization**. The frequentist penalty `λ‖β‖²` corresponds exactly to a Gaussian prior with `σ = 1/√λ`. More skeptical prior = larger λ = more shrinkage toward zero.

---

**Next**: In Chapter 11+, regularization becomes crucial for generalised linear models and multilevel models where priors on variance components control partial pooling.